In [1]:

# Cell 1: Install and verify the CUDA environment

!nvidia-smi
!pip install -q numba cupy-cuda12x pytest pynvml

import numpy as np
import numba
from numba import cuda
import cupy as cp

print("NumPy version:", np.__version__)
print("Numba version:", numba.__version__)
print("CuPy version:", cp.__version__)

if not cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Go to Runtime > Change runtime type > GPU."
    )

gpu = cuda.get_current_device()

print("\nCUDA environment is ready!")
print("GPU:", gpu.name)
print("Compute capability:", gpu.compute_capability)
print("CUDA available:", cuda.is_available())
print("CuPy GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])


Fri Oct 16 08:53:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/usr/local/lib/python3.13/dist-packages/cupy/_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


NumPy version: 2.1.3
Numba version: 0.61.2
CuPy version: 14.2.0

CUDA environment is ready!
GPU: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)
CUDA available: True
CuPy GPU: b'Tesla T4'


In [2]:

from numba import cuda
import numpy as np

@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)

    if i < out.size:
        out[i] = a[i] + b[i]


In [3]:

# Task 1: Test vector addition

a = np.array([1, 2, 3, 4, 5], dtype=np.float32)
b = np.array([10, 20, 30, 40, 50], dtype=np.float32)
out = np.zeros_like(a)

threads_per_block = 256
blocks_per_grid = (len(a) + threads_per_block - 1) // threads_per_block

vector_add_1d[blocks_per_grid, threads_per_block](a, b, out)

print("Array A:", a)
print("Array B:", b)
print("GPU result:", out)
print("Expected result:", a + b)

np.testing.assert_allclose(out, a + b, atol=1e-5)
print("TASK 1 PASSED!")



/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Array A: [1. 2. 3. 4. 5.]
Array B: [10. 20. 30. 40. 50.]
GPU result: [11. 22. 33. 44. 55.]
Expected result: [11. 22. 33. 44. 55.]
TASK 1 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [4]:

@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        out[i] = x[i] * alpha
        i += stride


In [5]:

# Task 2: Test grid-stride scaling

x = np.arange(1, 1001, dtype=np.float32)
alpha = np.float32(2.5)
out = np.zeros_like(x)

threads_per_block = 256
blocks_per_grid = 4

vector_scale_grid_stride[blocks_per_grid, threads_per_block](
    x, alpha, out
)

np.testing.assert_allclose(out, x * alpha, atol=1e-5)

print("First 10 input values:", x[:10])
print("First 10 output values:", out[:10])
print("TASK 2 PASSED!")


First 10 input values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]
First 10 output values: [ 2.5  5.   7.5 10.  12.5 15.  17.5 20.  22.5 25. ]
TASK 2 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [6]:

@cuda.jit
def matrix_add_2d(A, B, C):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


In [9]:

# Task 3:

A = np.arange(35, dtype=np.float32).reshape(5, 7)
B = np.ones((5, 7), dtype=np.float32) * 10
C = np.zeros_like(A)

threads = (16, 16)
blocks = (
    (A.shape[1] + threads[0] - 1) // threads[0],
    (A.shape[0] + threads[1] - 1) // threads[1]
)

matrix_add_2d[blocks, threads](A, B, C)

print("Result:\n", C)
np.testing.assert_allclose(C, A + B, atol=1e-5)
print("TASK 3 PASSED!")


Result:
 [[10. 11. 12. 13. 14. 15. 16.]
 [17. 18. 19. 20. 21. 22. 23.]
 [24. 25. 26. 27. 28. 29. 30.]
 [31. 32. 33. 34. 35. 36. 37.]
 [38. 39. 40. 41. 42. 43. 44.]]
TASK 3 PASSED!


In [11]:
#task 4

@cuda.jit
def matrix_transpose_naive(A, B):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]


In [12]:

# Task 4: Test matrix transpose

A = np.arange(15, dtype=np.float32).reshape(3, 5)
B = np.zeros((A.shape[1], A.shape[0]), dtype=np.float32)

threads = (16, 16)
blocks = (
    (A.shape[1] + 15) // 16,
    (A.shape[0] + 15) // 16
)

matrix_transpose_naive[blocks, threads](A, B)

print("Original shape:", A.shape)
print("Transposed shape:", B.shape)
print(B)

np.testing.assert_allclose(B, A.T, atol=1e-5)
print("TASK 4 PASSED!")


Original shape: (3, 5)
Transposed shape: (5, 3)
[[ 0.  5. 10.]
 [ 1.  6. 11.]
 [ 2.  7. 12.]
 [ 3.  8. 13.]
 [ 4.  9. 14.]]
TASK 4 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [13]:

@cuda.jit
def volume_init_3d(vol, D, H, W):
    x, y, z = cuda.grid(3)

    if z < D and y < H and x < W:
        vol[z, y, x] = z * 10000 + y * 100 + x


In [14]:

# Task 5: Test 3D volume initialization

D, H, W = 3, 4, 5
vol = np.zeros((D, H, W), dtype=np.int32)

threads = (8, 8, 8)
blocks = (
    (W + 7) // 8,
    (H + 7) // 8,
    (D + 7) // 8
)

volume_init_3d[blocks, threads](vol, D, H, W)

z, y, x = np.indices((D, H, W))
expected = z * 10000 + y * 100 + x

print("Volume shape:", vol.shape)
print("First 2D slice:\n", vol[0])

np.testing.assert_array_equal(vol, expected)
print("TASK 5 PASSED!")


Volume shape: (3, 4, 5)
First 2D slice:
 [[  0   1   2   3   4]
 [100 101 102 103 104]
 [200 201 202 203 204]
 [300 301 302 303 304]]
TASK 5 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [16]:

@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    y, x = cuda.grid(2)

    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        r = float(img_rgb[y, x, 0])
        g = float(img_rgb[y, x, 1])
        b = float(img_rgb[y, x, 2])

        img_gray[y, x] = 0.299 * r + 0.587 * g + 0.114 * b

# Task 6: Test RGB-to-grayscale conversion

img_rgb = np.array([
    [[255, 0, 0], [0, 255, 0]],
    [[0, 0, 255], [255, 255, 255]]
], dtype=np.uint8)

img_gray = np.zeros(img_rgb.shape[:2], dtype=np.float32)

threads = (16, 16)
blocks = (
    (img_rgb.shape[0] + 15) // 16,
    (img_rgb.shape[1] + 15) // 16
)

rgb_to_grayscale[blocks, threads](img_rgb, img_gray)

expected = (
    0.299 * img_rgb[:, :, 0].astype(np.float32)
    + 0.587 * img_rgb[:, :, 1].astype(np.float32)
    + 0.114 * img_rgb[:, :, 2].astype(np.float32)
)

print("Grayscale image:\n", img_gray)

np.testing.assert_allclose(img_gray, expected, atol=1e-4)
print("TASK 6 PASSED!")


Grayscale image:
 [[ 76.245 149.685]
 [ 29.07  255.   ]]
TASK 6 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [17]:

@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)

    if i < out.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]


In [19]:

# Task 7: Test linear interpolation

a = np.array([0, 10, 20, 30], dtype=np.float32)
b = np.array([10, 20, 40, 50], dtype=np.float32)
t = np.array([0, 0.25, 0.5, 1], dtype=np.float32)
out = np.zeros_like(a)

threads = 256
blocks = (len(a) + threads - 1) // threads

vector_lerp[blocks, threads](a, b, t, out)

expected = (1 - t) * a + t * b

print("LERP output:", out)

np.testing.assert_allclose(out, expected, atol=1e-5)
print("TASK 7 PASSED!")


LERP output: [ 0.  12.5 30.  50. ]
TASK 7 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [20]:

@cuda.jit
def vector_threshold(x, thresh, mask):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        if x[i] >= thresh:
            mask[i] = 1
        else:
            mask[i] = 0

        i += stride


In [21]:

# Task 8: Test thresholding

x = np.array([2, 5, 7, 1, 9, 4, 5], dtype=np.float32)
threshold = np.float32(5)
mask = np.zeros(x.size, dtype=np.uint8)

threads = 256
blocks = 1

vector_threshold[blocks, threads](x, threshold, mask)

expected = (x >= threshold).astype(np.uint8)

print("Input:", x)
print("Mask:", mask)

np.testing.assert_array_equal(mask, expected)
print("TASK 8 PASSED!")


Input: [2. 5. 7. 1. 9. 4. 5.]
Mask: [0 1 1 0 1 0 1]
TASK 8 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [22]:

@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    i = cuda.grid(1)

    if i < out_arr.size:
        out_arr[i] = in_arr[i * stride]


In [23]:

# Task 9: Test different access strides

N = 1024 * 1024
x = np.arange(N, dtype=np.float32)

threads = 256

for stride in [1, 2, 4, 8, 16, 32]:
    count = (N + stride - 1) // stride
    out = np.zeros(count, dtype=np.float32)
    blocks = (count + threads - 1) // threads

    benchmark_strided_access[blocks, threads](x, out, stride)

    expected = x[::stride]
    np.testing.assert_array_equal(out, expected)

    print(f"Stride {stride}: correctness PASSED, elements = {count}")

print("TASK 9 CORRECTNESS TESTS PASSED!")


Stride 1: correctness PASSED, elements = 1048576
Stride 2: correctness PASSED, elements = 524288
Stride 4: correctness PASSED, elements = 262144
Stride 8: correctness PASSED, elements = 131072
Stride 16: correctness PASSED, elements = 65536
Stride 32: correctness PASSED, elements = 32768
TASK 9 CORRECTNESS TESTS PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [25]:

@cuda.jit
def stencil_1d_shared(in_arr, out_arr, radius):
    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    BLOCK_SIZE = 256
    MAX_RADIUS = 128

    # Shared memory includes the block and halo space.
    tile = cuda.shared.array(512, dtype=numba.float32)

    # Cooperatively load the tile and halo.
    for s in range(tid, BLOCK_SIZE + 2 * MAX_RADIUS, BLOCK_SIZE):
        idx = cuda.blockIdx.x * BLOCK_SIZE + s - MAX_RADIUS

        if idx < 0:
            idx = 0
        elif idx >= in_arr.size:
            idx = in_arr.size - 1

        tile[s] = in_arr[idx]

    cuda.syncthreads()

    if i < in_arr.size:
        total = 0.0

        for k in range(-radius, radius + 1):
            total += tile[tid + MAX_RADIUS + k]

        out_arr[i] = total / (2 * radius + 1)


In [26]:

# Task 10: Test the moving average

from scipy.ndimage import uniform_filter1d

x = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    dtype=np.float32
)
out = np.zeros_like(x)

radius = 2
threads = 256
blocks = (len(x) + threads - 1) // threads

stencil_1d_shared[blocks, threads](x, out, radius)

expected = uniform_filter1d(
    x, size=2 * radius + 1, mode="nearest"
)

print("GPU result:", out)
print("Expected:", expected)

np.testing.assert_allclose(out, expected, atol=1e-4)
print("TASK 10 PASSED!")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU result: [1.6 2.2 3.  4.  5.  6.  7.  8.  8.8 9.4]
Expected: [1.6 2.2 3.  4.  5.  6.  7.  8.  8.8 9.4]
TASK 10 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [33]:

@cuda.jit
def conv_1d_shared_tiled(x, kernel, y):
    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    BLOCK_SIZE = 256
    MAX_RADIUS = 128

    tile = cuda.shared.array(512, dtype=numba.float32)
    radius = kernel.size // 2

    # Load the input tile and halo into shared memory.
    # IMPORTANT: convolution uses zero-padding outside the input.
    for s in range(tid, BLOCK_SIZE + 2 * MAX_RADIUS, BLOCK_SIZE):
        idx = cuda.blockIdx.x * BLOCK_SIZE + s - MAX_RADIUS

        if idx < 0 or idx >= x.size:
            tile[s] = 0.0
        else:
            tile[s] = x[idx]

    cuda.syncthreads()

    if i < y.size:
        total = 0.0

        for k in range(kernel.size):
            total += tile[tid + MAX_RADIUS + radius - k] * kernel[k]

        y[i] = total


In [34]:

# Task 11: Test 1D convolution

x = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8],
    dtype=np.float32
)
kernel = np.array([0.25, 0.5, 0.25], dtype=np.float32)
y = np.zeros_like(x)

threads = 256
blocks = (len(x) + threads - 1) // threads

conv_1d_shared_tiled[blocks, threads](x, kernel, y)

expected = np.convolve(x, kernel, mode="same")

print("GPU result:", y)
print("Expected:", expected)

np.testing.assert_allclose(y, expected, atol=1e-4)
print("TASK 11 PASSED!")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU result: [1.   2.   3.   4.   5.   6.   7.   5.75]
Expected: [1.   2.   3.   4.   5.   6.   7.   5.75]
TASK 11 PASSED!


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [35]:

@cuda.jit
def matmul_tiled_shared(A, B, C):
    TILE = 16

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * TILE + ty
    col = cuda.blockIdx.x * TILE + tx

    shared_A = cuda.shared.array((16, 16), dtype=numba.float32)
    shared_B = cuda.shared.array((16, 16), dtype=numba.float32)

    total = 0.0
    num_tiles = (A.shape[1] + TILE - 1) // TILE

    for t in range(num_tiles):
        a_col = t * TILE + tx
        b_row = t * TILE + ty

        if row < A.shape[0] and a_col < A.shape[1]:
            shared_A[ty, tx] = A[row, a_col]
        else:
            shared_A[ty, tx] = 0.0

        if b_row < B.shape[0] and col < B.shape[1]:
            shared_B[ty, tx] = B[b_row, col]
        else:
            shared_B[ty, tx] = 0.0

        cuda.syncthreads()

        for k in range(TILE):
            total += shared_A[ty, k] * shared_B[k, tx]

        cuda.syncthreads()

    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = total


In [36]:

rng = np.random.default_rng(12)

A = rng.random((32, 17), dtype=np.float32)
B = rng.random((17, 19), dtype=np.float32)
C = np.empty((32, 19), dtype=np.float32)

dA = cuda.to_device(A)
dB = cuda.to_device(B)
dC = cuda.to_device(C)

threads = (16, 16)
blocks = ((C.shape[1] + 15) // 16, (C.shape[0] + 15) // 16)

matmul_tiled_shared[blocks, threads](dA, dB, dC)
result = dC.copy_to_host()
expected = A @ B

np.testing.assert_allclose(result, expected, rtol=1e-4, atol=1e-4)
print("TASK 12 PASSED!")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


TASK 12 PASSED!


In [37]:

@cuda.jit
def transpose_bank_conflict_free(A, B):
    TILE = 32

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty

    tile = cuda.shared.array((32, 33), dtype=numba.float32)

    if y < A.shape[0] and x < A.shape[1]:
        tile[ty, tx] = A[y, x]

    cuda.syncthreads()

    # Read the shared tile transposed.
    out_col = cuda.blockIdx.y * TILE + tx
    out_row = cuda.blockIdx.x * TILE + ty

    if out_row < B.shape[0] and out_col < B.shape[1]:
        B[out_row, out_col] = tile[tx, ty]


In [41]:
#Task 13
A = np.arange(37 * 53, dtype=np.float32).reshape(37, 53)
B = np.empty((53, 37), dtype=np.float32)

dA = cuda.to_device(A)
dB = cuda.to_device(B)

threads = (32, 32)
blocks = ((A.shape[1] + 31) // 32, (A.shape[0] + 31) // 32)

transpose_bank_conflict_free[blocks, threads](dA, dB)
result = dB.copy_to_host()

np.testing.assert_array_equal(result, A.T)
print("TASK 13 PASSED!")


TASK 13 PASSED!


In [42]:

@cuda.jit
def reduction_sum_shared(in_arr, block_sums):
    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    shared = cuda.shared.array(256, dtype=numba.float32)

    if i < in_arr.size:
        shared[tid] = in_arr[i]
    else:
        shared[tid] = 0.0

    cuda.syncthreads()

    stride = 128
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        block_sums[cuda.blockIdx.x] = shared[0]


In [43]:

rng = np.random.default_rng(14)
x = rng.random(1003, dtype=np.float32)

dX = cuda.to_device(x)
num_blocks = (x.size + 255) // 256
dSums = cuda.device_array(num_blocks, dtype=np.float32)

reduction_sum_shared[num_blocks, 256](dX, dSums)

partial_sums = dSums.copy_to_host()
result = partial_sums.sum()
expected = x.sum(dtype=np.float32)

np.testing.assert_allclose(result, expected, rtol=1e-4, atol=1e-4)
print("TASK 14 PASSED!")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


TASK 14 PASSED!


In [44]:

@cuda.jit
def reduction_argmax_shared(in_arr, block_max_vals, block_max_indices):
    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    values = cuda.shared.array(256, dtype=numba.float32)
    indices = cuda.shared.array(256, dtype=numba.int32)

    if i < in_arr.size:
        values[tid] = in_arr[i]
        indices[tid] = i
    else:
        values[tid] = -float("inf")
        indices[tid] = 2147483647

    cuda.syncthreads()

    stride = 128
    while stride > 0:
        if tid < stride:
            right_value = values[tid + stride]
            right_index = indices[tid + stride]

            if (right_value > values[tid] or
                (right_value == values[tid] and right_index < indices[tid])):
                values[tid] = right_value
                indices[tid] = right_index

        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        block_max_vals[cuda.blockIdx.x] = values[0]
        block_max_indices[cuda.blockIdx.x] = indices[0]


In [45]:

rng = np.random.default_rng(15)
x = rng.random(1003, dtype=np.float32)
x[777] = 2.0  # Ensure the maximum is unambiguous.

dX = cuda.to_device(x)
num_blocks = (x.size + 255) // 256

dVals = cuda.device_array(num_blocks, dtype=np.float32)
dIndices = cuda.device_array(num_blocks, dtype=np.int32)

reduction_argmax_shared[num_blocks, 256](dX, dVals, dIndices)

block_vals = dVals.copy_to_host()
block_indices = dIndices.copy_to_host()

best_block = np.argmax(block_vals)
result_value = block_vals[best_block]
result_index = block_indices[best_block]

assert result_value == x.max()
assert result_index == int(np.argmax(x))
print("TASK 15 PASSED!")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


TASK 15 PASSED!


In [46]:

FILTER_WEIGHTS = np.full(64, 1.0 / 64, dtype=np.float32)


In [47]:

@cuda.jit
def constant_memory_filter(signal, out):
    weights = cuda.const.array_like(FILTER_WEIGHTS)

    i = cuda.grid(1)

    if i < out.size:
        total = 0.0
        radius = 32

        for k in range(64):
            # Center the 64-tap filter around the output position.
            idx = i + k - radius

            if 0 <= idx < signal.size:
                total += signal[idx] * weights[k]

        out[i] = total


In [48]:

rng = np.random.default_rng(16)
x = rng.random(256, dtype=np.float32)
y = np.empty_like(x)

dX = cuda.to_device(x)
dY = cuda.device_array_like(dX)

threads = 128
blocks = (x.size + threads - 1) // threads

constant_memory_filter[blocks, threads](dX, dY)
result = dY.copy_to_host()

# CPU reference with the same zero-padding and alignment.
expected = np.zeros_like(x)
for i in range(x.size):
    for k in range(64):
        idx = i + k - 32
        if 0 <= idx < x.size:
            expected[i] += x[idx] * FILTER_WEIGHTS[k]

np.testing.assert_allclose(result, expected, rtol=1e-4, atol=1e-4)
print("TASK 16 PASSED!")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


TASK 16 PASSED!


In [49]:

import time

def benchmark_pinned_transfer(size_bytes):
    n = max(1, size_bytes // np.dtype(np.float32).itemsize)

    # Ordinary pageable host memory.
    pageable = np.ones(n, dtype=np.float32)

    # Page-locked (pinned) host memory.
    pinned = cuda.pinned_array(n, dtype=np.float32)
    pinned[:] = 1.0

    # Allocate GPU buffers once so allocation time isn't included.
    d_pageable = cuda.device_array(n, dtype=np.float32)
    d_pinned = cuda.device_array(n, dtype=np.float32)

    # Warm up both paths.
    d_pageable.copy_to_device(pageable)
    cuda.synchronize()
    d_pinned.copy_to_device(pinned)
    cuda.synchronize()

    # Host-to-device: pageable memory.
    start = time.perf_counter()
    d_pageable.copy_to_device(pageable)
    cuda.synchronize()
    pageable_h2d = time.perf_counter() - start

    # Host-to-device: pinned memory.
    start = time.perf_counter()
    d_pinned.copy_to_device(pinned)
    cuda.synchronize()
    pinned_h2d = time.perf_counter() - start

    # Device-to-host: pageable memory.
    start = time.perf_counter()
    result_pageable = d_pageable.copy_to_host()
    cuda.synchronize()
    pageable_d2h = time.perf_counter() - start

    # Device-to-host: pinned memory.
    start = time.perf_counter()
    d_pinned.copy_to_host(pinned)
    cuda.synchronize()
    pinned_d2h = time.perf_counter() - start

    return {
        "size_bytes": n * 4,
        "pageable_h2d_seconds": pageable_h2d,
        "pinned_h2d_seconds": pinned_h2d,
        "pageable_d2h_seconds": pageable_d2h,
        "pinned_d2h_seconds": pinned_d2h,
    }


In [50]:

results = benchmark_pinned_transfer(16 * 1024 * 1024)

for name, value in results.items():
    print(f"{name}: {value}")

assert results["size_bytes"] == 16 * 1024 * 1024
print("TASK 17 RAN SUCCESSFULLY!")


size_bytes: 16777216
pageable_h2d_seconds: 0.008856162000029144
pinned_h2d_seconds: 0.003416174999983923
pageable_d2h_seconds: 0.019087167999941812
pinned_d2h_seconds: 0.0015825599998606776
TASK 17 RAN SUCCESSFULLY!
